# Disaggregasi Data Tanaman Pangan Kabupaten Kediri → Kecamatan

Notebook ini menjalankan rencana disaggregasi yang kamu pilih:

**Tanaman Pangan Kabupaten Kediri + Luas Sawah per Kecamatan → Proporsi Luas Sawah → Estimasi Luas Panen & Produksi → Produktivitas → Validasi → Dataset Final.**

Target output:
- 26 kecamatan × 5 komoditas = **130 baris**
- Estimasi luas panen per kecamatan
- Estimasi produksi per kecamatan
- Produktivitas hasil estimasi
- Validasi total terhadap data Kabupaten
- Metadata bahwa data kecamatan adalah **Estimated**
- Export Excel dan CSV

**Catatan penting:** program tidak membuat variasi produktivitas secara acak. Karena luas panen dan produksi sama-sama dibagi menggunakan proporsi luas sawah, produktivitas komoditas yang sama akan sama antar kecamatan. Ini adalah konsekuensi matematis dari metode yang kamu pilih.


In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("✓ Program siap dijalankan.")


In [ ]:
# ============================================================
# 1. LOKASI FILE
# ============================================================

FILE_TANAMAN = "Tanaman Pangan.xlsx"
FILE_SAWAH = "Luas Lahan  Per Kecamatan.csv"

if not Path(FILE_TANAMAN).exists():
    raise FileNotFoundError(f"File tidak ditemukan: {FILE_TANAMAN}")

if not Path(FILE_SAWAH).exists():
    raise FileNotFoundError(f"File tidak ditemukan: {FILE_SAWAH}")

print("✓ Kedua file ditemukan.")


In [ ]:
# ============================================================
# 2. BACA DATA TANAMAN PANGAN KABUPATEN
# ============================================================

df_tanaman_raw = pd.read_excel(FILE_TANAMAN)

print("Ukuran dataset:", df_tanaman_raw.shape)
display(df_tanaman_raw)


In [ ]:
# ============================================================
# 3. CLEANING DATA TANAMAN PANGAN
# ============================================================

def extract_number(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    value = re.sub(r"[^0-9.,-]", "", value)

    if "," in value and "." in value:
        value = value.replace(",", "")
    elif "," in value:
        value = value.replace(",", ".")

    try:
        return float(value)
    except ValueError:
        return np.nan


required_crop_columns = {"Tanaman", "Luas Panen", "Produksi", "Produktivitas"}
missing_crop = required_crop_columns - set(df_tanaman_raw.columns)

if missing_crop:
    raise ValueError(f"Kolom tanaman pangan tidak ditemukan: {missing_crop}")

df_tanaman = df_tanaman_raw.copy()

df_tanaman["luas_panen_kab_ha"] = df_tanaman["Luas Panen"].apply(extract_number)
df_tanaman["produksi_kab_ton"] = df_tanaman["Produksi"].apply(extract_number)
df_tanaman["produktivitas_kab_kuintal_ha"] = (
    df_tanaman["Produktivitas"].apply(extract_number)
)

df_tanaman = df_tanaman[
    [
        "Tanaman",
        "luas_panen_kab_ha",
        "produksi_kab_ton",
        "produktivitas_kab_kuintal_ha",
    ]
].copy()

df_tanaman["Tanaman"] = df_tanaman["Tanaman"].astype(str).str.strip()

if df_tanaman[["luas_panen_kab_ha", "produksi_kab_ton"]].isna().any().any():
    raise ValueError("Ada nilai luas panen/produksi yang tidak berhasil dibaca sebagai angka.")

print("Komoditas:", df_tanaman["Tanaman"].tolist())
display(df_tanaman)


In [ ]:
# ============================================================
# 4. BACA DATA LUAS SAWAH PER KECAMATAN
# ============================================================

# CSV yang kamu upload memiliki 4 baris pembuka sebelum header data.
df_sawah_raw = pd.read_csv(FILE_SAWAH, skiprows=4)

print("Ukuran dataset:", df_sawah_raw.shape)
display(df_sawah_raw.head())


In [ ]:
# ============================================================
# 5. CLEANING DATA LUAS SAWAH
# ============================================================

if len(df_sawah_raw.columns) < 5:
    raise ValueError("Struktur CSV luas sawah tidak sesuai.")

df_sawah = df_sawah_raw.iloc[:, :5].copy()

df_sawah.columns = [
    "kecamatan",
    "luas_sawah_ha",
    "lahan_pertanian_non_sawah_ha",
    "lahan_non_pertanian_ha",
    "total_lahan_ha",
]

def clean_kecamatan(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    return re.sub(r"^\[\d+\]\s*", "", value).strip()

df_sawah["kecamatan"] = df_sawah["kecamatan"].apply(clean_kecamatan)

numeric_cols = [
    "luas_sawah_ha",
    "lahan_pertanian_non_sawah_ha",
    "lahan_non_pertanian_ha",
    "total_lahan_ha",
]

for col in numeric_cols:
    df_sawah[col] = pd.to_numeric(df_sawah[col], errors="coerce")

if df_sawah["kecamatan"].isna().any():
    raise ValueError("Ada nama kecamatan yang kosong.")

if df_sawah["luas_sawah_ha"].isna().any():
    raise ValueError("Ada nilai luas sawah yang tidak valid.")

display(df_sawah)


In [ ]:
# ============================================================
# 6. VALIDASI KECAMATAN DAN TOTAL LUAS SAWAH
# ============================================================

jumlah_kecamatan = df_sawah["kecamatan"].nunique()

if jumlah_kecamatan != 26:
    raise ValueError(
        f"Jumlah kecamatan = {jumlah_kecamatan}, bukan 26. "
        "Periksa dataset luas sawah."
    )

if df_sawah["kecamatan"].duplicated().any():
    raise ValueError("Ada nama kecamatan yang duplikat.")

total_luas_sawah = df_sawah["luas_sawah_ha"].sum()

print(f"Jumlah kecamatan : {jumlah_kecamatan}")
print(f"Total luas sawah : {total_luas_sawah:,.2f} Ha")

if np.isclose(total_luas_sawah, 44066, atol=1):
    print("✓ Total luas sawah sesuai sekitar 44.066 Ha.")
else:
    print("⚠ Total luas sawah berbeda dari 44.066 Ha. Program tetap menggunakan total dari file.")


In [ ]:
# ============================================================
# 7. HITUNG PROPORSI LUAS SAWAH
# ============================================================

df_sawah["proporsi_sawah"] = (
    df_sawah["luas_sawah_ha"] / total_luas_sawah
)

df_sawah["persentase_sawah"] = (
    df_sawah["proporsi_sawah"] * 100
)

total_proporsi = df_sawah["proporsi_sawah"].sum()

print(f"Total proporsi = {total_proporsi:.10f}")

if not np.isclose(total_proporsi, 1.0):
    raise ValueError("Total proporsi luas sawah tidak sama dengan 1.")

display(
    df_sawah[
        ["kecamatan", "luas_sawah_ha", "proporsi_sawah", "persentase_sawah"]
    ].sort_values("luas_sawah_ha", ascending=False)
)


In [ ]:
# ============================================================
# 8. BENTUK 26 KECAMATAN × 5 KOMODITAS
# ============================================================

df_kecamatan = df_sawah[
    [
        "kecamatan",
        "luas_sawah_ha",
        "proporsi_sawah",
        "persentase_sawah",
    ]
].copy()

df_final = df_kecamatan.merge(df_tanaman, how="cross")

expected_rows = jumlah_kecamatan * len(df_tanaman)

if len(df_final) != expected_rows:
    raise ValueError(
        f"Jumlah baris {len(df_final)} tidak sesuai target {expected_rows}."
    )

print(
    f"✓ Dataset terbentuk: {jumlah_kecamatan} kecamatan × "
    f"{len(df_tanaman)} komoditas = {len(df_final)} baris"
)


In [ ]:
# ============================================================
# 9. ESTIMASI LUAS PANEN DAN PRODUKSI
# ============================================================

df_final["luas_panen_estimasi_ha"] = (
    df_final["luas_panen_kab_ha"] *
    df_final["proporsi_sawah"]
)

df_final["produksi_estimasi_ton"] = (
    df_final["produksi_kab_ton"] *
    df_final["proporsi_sawah"]
)

# Ton/Ha -> Kuintal/Ha
df_final["produktivitas_estimasi_kuintal_ha"] = np.where(
    df_final["luas_panen_estimasi_ha"] > 0,
    (
        df_final["produksi_estimasi_ton"] /
        df_final["luas_panen_estimasi_ha"]
    ) * 10,
    np.nan,
)

display(df_final.head(10))


In [ ]:
# ============================================================
# 10. METADATA DATA ESTIMASI
# ============================================================

df_final["tahun_data"] = 2023
df_final["status_data"] = "Estimated"
df_final["metode_estimasi"] = "Proporsi Luas Sawah Kecamatan"
df_final["sumber_data"] = (
    "Tanaman Pangan Kabupaten Kediri + Luas Sawah per Kecamatan"
)
df_final["catatan"] = (
    "Estimasi tingkat kecamatan berdasarkan proporsi luas sawah; "
    "bukan data observasi aktual kecamatan."
)

kolom_final = [
    "tahun_data",
    "kecamatan",
    "Tanaman",
    "luas_sawah_ha",
    "proporsi_sawah",
    "persentase_sawah",
    "luas_panen_kab_ha",
    "produksi_kab_ton",
    "produktivitas_kab_kuintal_ha",
    "luas_panen_estimasi_ha",
    "produksi_estimasi_ton",
    "produktivitas_estimasi_kuintal_ha",
    "status_data",
    "metode_estimasi",
    "sumber_data",
    "catatan",
]

df_final = df_final[kolom_final]

display(df_final.head(15))


In [ ]:
# ============================================================
# 11. VALIDASI TOTAL PER KOMODITAS
# ============================================================

validasi = (
    df_final.groupby("Tanaman")
    .agg(
        luas_panen_kabupaten=("luas_panen_kab_ha", "first"),
        total_luas_panen_estimasi=("luas_panen_estimasi_ha", "sum"),
        produksi_kabupaten=("produksi_kab_ton", "first"),
        total_produksi_estimasi=("produksi_estimasi_ton", "sum"),
    )
    .reset_index()
)

validasi["selisih_luas_panen"] = (
    validasi["total_luas_panen_estimasi"] -
    validasi["luas_panen_kabupaten"]
)

validasi["selisih_produksi"] = (
    validasi["total_produksi_estimasi"] -
    validasi["produksi_kabupaten"]
)

validasi["luas_panen_valid"] = np.isclose(
    validasi["total_luas_panen_estimasi"],
    validasi["luas_panen_kabupaten"],
    rtol=1e-9,
    atol=1e-8,
)

validasi["produksi_valid"] = np.isclose(
    validasi["total_produksi_estimasi"],
    validasi["produksi_kabupaten"],
    rtol=1e-9,
    atol=1e-8,
)

display(validasi)


In [ ]:
# ============================================================
# 12. CEK PRODUKTIVITAS
# ============================================================

cek_produktivitas = (
    df_final.groupby("Tanaman")["produktivitas_estimasi_kuintal_ha"]
    .agg(
        minimum="min",
        maksimum="max",
        rata_rata="mean",
        standar_deviasi="std",
    )
    .reset_index()
)

print(
    "Catatan: dengan metode ini, produktivitas komoditas yang sama "
    "akan sama antar kecamatan."
)

display(cek_produktivitas)


In [ ]:
# ============================================================
# 13. VALIDASI AKHIR
# ============================================================

assert df_final["kecamatan"].nunique() == 26
assert df_final["Tanaman"].nunique() == 5
assert len(df_final) == 130
assert validasi["luas_panen_valid"].all()
assert validasi["produksi_valid"].all()

print("========== VALIDASI AKHIR ==========")
print(f"✓ Kecamatan : {df_final['kecamatan'].nunique()}")
print(f"✓ Komoditas : {df_final['Tanaman'].nunique()}")
print(f"✓ Baris     : {len(df_final)}")
print("✓ Total luas panen kembali ke total Kabupaten.")
print("✓ Total produksi kembali ke total Kabupaten.")
print("✓ Dataset final siap digunakan.")


In [ ]:
# ============================================================
# 14. EXPORT KE EXCEL
# ============================================================

OUTPUT_EXCEL = "Dataset_Tanaman_Pangan_Kecamatan_Kediri_Estimasi.xlsx"

with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    df_final.to_excel(
        writer,
        sheet_name="Data_Final",
        index=False,
    )
    validasi.to_excel(
        writer,
        sheet_name="Validasi_Total",
        index=False,
    )
    df_sawah.to_excel(
        writer,
        sheet_name="Proporsi_Sawah",
        index=False,
    )
    df_tanaman.to_excel(
        writer,
        sheet_name="Data_Kabupaten",
        index=False,
    )

print(f"✓ Excel berhasil dibuat: {OUTPUT_EXCEL}")


In [ ]:
# ============================================================
# 15. EXPORT KE CSV
# ============================================================

OUTPUT_CSV = "Dataset_Tanaman_Pangan_Kecamatan_Kediri_Estimasi.csv"

df_final.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig",
)

print(f"✓ CSV berhasil dibuat: {OUTPUT_CSV}")


In [ ]:
# ============================================================
# 16. TAMPILKAN HASIL AKHIR
# ============================================================

display(
    df_final[
        [
            "tahun_data",
            "kecamatan",
            "Tanaman",
            "luas_sawah_ha",
            "proporsi_sawah",
            "luas_panen_estimasi_ha",
            "produksi_estimasi_ton",
            "produktivitas_estimasi_kuintal_ha",
            "status_data",
        ]
    ].head(20)
)

print("\nFile output:")
print("-", OUTPUT_EXCEL)
print("-", OUTPUT_CSV)
